# 🛒 Vietnamese Graph RAG — Part 1
## Data · Preprocessing · NER · So sánh Embedding (TF-IDF / Word2Vec / GloVe-SVD / PhoBERT)

**Đầu ra:** `data/train_processed.csv`, `results_part1.json`, các hình PNG.

> ⚙️ Bật **Internet** + **GPU (T4)** trong Settings của Kaggle trước khi chạy.
> So sánh embedding bằng **Precision@k & MRR** (nhãn aspect là ground-truth) — không dùng "avg cosine".

## 0. Cài đặt thư viện

In [ ]:
!pip install -q underthesea gensim networkx
!pip install -q transformers

In [ ]:
import os, re, json, warnings, urllib.request, zipfile, glob
import numpy as np
import pandas as pd
DATA_DIR = 'data'; os.makedirs(DATA_DIR, exist_ok=True)

def _find_csv(name):
    hits = glob.glob('/kaggle/input/**/' + name, recursive=True)
    if hits: return hits[0]
    for p in [DATA_DIR + '/' + name, DATA_DIR + '/raw/' + name, name]:
        if os.path.exists(p): return p
    return None

def load_split(name):
    p = _find_csv(name)
    if p is None:
        zp = DATA_DIR + '/UIT-ViSFD.zip'
        if not os.path.exists(zp):
            try:
                print('Downloading UIT-ViSFD.zip ...')
                urllib.request.urlretrieve('https://github.com/LuongPhan/UIT-ViSFD/raw/main/UIT-ViSFD.zip', zp)
            except Exception as e:
                print('zip download failed:', e)
        if os.path.exists(zp):
            try:
                with zipfile.ZipFile(zp) as z: z.extractall(DATA_DIR)
            except Exception as e:
                print('extract failed:', e)
        p = _find_csv(name)
    if p is None:
        p = DATA_DIR + '/' + name
        urllib.request.urlretrieve('https://raw.githubusercontent.com/kimkim00/UIT-ViSFD/main/' + name, p)
    return pd.read_csv(p)

import matplotlib.pyplot as plt
from collections import Counter
warnings.filterwarnings('ignore')
try: plt.style.use('seaborn-v0_8-whitegrid')
except Exception: pass
np.random.seed(42)
print('Libraries loaded!')

## 1. Load dữ liệu UIT-ViSFD

In [ ]:
train_df = load_split('Train.csv')
dev_df   = load_split('Dev.csv')
test_df  = load_split('Test.csv')
print(f'Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}')
print('Columns:', list(train_df.columns))
train_df.head(3)

In [ ]:
# FIX #1: regex giu ky tu '&' -> KHONG mat aspect SER&ACC (1995 lan trong train)
ASPECT_PATTERN = re.compile(r'\{([\w&]+)#(\w+)\}')

def parse_label(label_str):
    """'{CAMERA#Positive};{SER&ACC#Negative};{OTHERS};' -> [('CAMERA','Positive'),('SER&ACC','Negative')]"""
    if pd.isna(label_str):
        return []
    return [(m.group(1), m.group(2)) for m in ASPECT_PATTERN.finditer(str(label_str))]

for df in (train_df, dev_df, test_df):
    df['parsed_labels'] = df['label'].apply(parse_label)
    df['aspects']    = df['parsed_labels'].apply(lambda x: [a for a, s in x])
    df['sentiments'] = df['parsed_labels'].apply(lambda x: [s for a, s in x])

aspect_counts = Counter(a for lst in train_df['aspects'] for a in lst)
print('Phan bo aspect (train) — SER&ACC gio da xuat hien:')
for a, c in aspect_counts.most_common():
    print(f'  {a:12s}: {c}')
assert 'SER&ACC' in aspect_counts, "SER&ACC van bi mat - kiem tra regex!"


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
asp_df = pd.DataFrame(aspect_counts.most_common(), columns=['Aspect', 'Count'])
axes[0].barh(asp_df['Aspect'][::-1], asp_df['Count'][::-1], color='teal')
axes[0].set_title('Phan bo Aspect (train)')
if 'n_star' in train_df.columns:
    train_df['n_star'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Phan bo so sao'); axes[1].set_xlabel('Stars')
plt.tight_layout(); plt.savefig('fig_aspect_distribution.png', dpi=150, bbox_inches='tight'); plt.show()

## 2. Tiền xử lý tiếng Việt

In [ ]:
from underthesea import word_tokenize
STOPWORDS = set('va cua la co cho duoc trong voi khong nay cac mot nhung da khi de tu cung nhu nhung hay hoac vi nen thi ma rat lai bi do neu ve theo tai den con se dang ra vao len toi minh a'.split())
STOPWORDS |= set('và của là có cho được trong với không này các một những đã khi để từ cũng như nhưng hay hoặc vì nên thì mà rất lại bị do nếu về theo tại đến còn sẽ đang ra vào lên tôi mình ạ'.split())

URL_RE = re.compile(r'http\S+|www\S+'); NONWORD_RE = re.compile(r'[^\w\s]')
NUM_RE = re.compile(r'\d+');           SPACE_RE = re.compile(r'\s+')

def preprocess_vietnamese(text):
    if pd.isna(text): return ''
    t = URL_RE.sub(' ', str(text).lower().strip())
    t = NONWORD_RE.sub(' ', t); t = NUM_RE.sub(' ', t); t = SPACE_RE.sub(' ', t).strip()
    toks = word_tokenize(t, format='text').split()
    return ' '.join(w for w in toks if w not in STOPWORDS and len(w) > 1)

# PhoBERT nen nhan input da TACH TU (giu dau, giu so) -> seg() rieng
def seg(text):
    return word_tokenize(str(text)[:256], format='text')

for df in (train_df, dev_df, test_df):
    df['clean_text'] = df['comment'].apply(preprocess_vietnamese)
print('Preprocess xong.')
print('VD :', str(train_df.iloc[0]['comment'])[:80])
print(' ->', train_df.iloc[0]['clean_text'][:80])

## 3. Bảng ánh xạ từ-khóa tiếng Việt → aspect (dùng lại ở Part 2)

In [ ]:
ASPECTS = ['SCREEN','CAMERA','BATTERY','PERFORMANCE','STORAGE','DESIGN','PRICE','GENERAL','FEATURES','SER&ACC']

# FIX #3: bang anh xa tu-khoa tieng Viet -> aspect code (kich hoat graph context cho cau hoi tieng Viet)
ASPECT_KEYWORDS = {
    'SCREEN':      ['man hinh','màn hình','hien thi','hiển thị','cam ung','cảm ứng','tan so quet','do phan giai','độ phân giải'],
    'CAMERA':      ['camera','chup','chụp','anh','ảnh','quay','selfie','chup dem','chụp đêm','zoom','ong kinh','ống kính'],
    'BATTERY':     ['pin','sac','sạc','dung luong pin','trau','trâu','tut pin','tụt pin','chai pin'],
    'PERFORMANCE': ['hieu nang','hiệu năng','muot','mượt','lag','giat','giật','chip','ram','cau hinh','cấu hình','xu ly','chay','chạy','nhanh'],
    'STORAGE':     ['bo nho','bộ nhớ','luu tru','lưu trữ','dung luong','dung lượng','rom','gb','day bo nho'],
    'DESIGN':      ['thiet ke','thiết kế','kieu dang','kiểu dáng','dep','đẹp','mong','mỏng','cam','cầm','chat lieu','mau sac'],
    'PRICE':       ['gia','giá','tien','tiền','re','rẻ','dat','đắt','mac','mắc','hop ly','hợp lý','tam gia','gia thanh'],
    'FEATURES':    ['tinh nang','tính năng','loa','am thanh','âm thanh','wifi','song','sóng','bluetooth','van tay','vân tay','bao mat','nfc','cam bien','cảm biến'],
    'SER&ACC':     ['dich vu','dịch vụ','bao hanh','bảo hành','phu kien','phụ kiện','nhan vien','nhân viên','tu van','tư vấn','giao hang','giao hàng','shop','cua hang','cửa hàng','dong goi','đóng gói'],
    'GENERAL':     ['san pham','sản phẩm','may','máy','dien thoai','điện thoại','tong the','tổng thể','noi chung','on dinh','ổn'],
}

def aspects_from_text(text):
    """Aspect QUAN SAT duoc trong noi dung (tin hieu doc lap voi nhan vang)."""
    t = str(text).lower()
    return {a for a, kws in ASPECT_KEYWORDS.items() if any(k in t for k in kws)}

def aspect_from_query(query):
    """Aspect chinh ma cau hoi nham toi (nhieu keyword khop nhat)."""
    t = str(query).lower(); best, bn = None, 0
    for a, kws in ASPECT_KEYWORDS.items():
        n = sum(1 for k in kws if k in t)
        if n > bn: best, bn = a, n
    return best

BRAND_GAZETTEER = {
    'iphone':'Apple','apple':'Apple','samsung':'Samsung','galaxy':'Samsung','xiaomi':'Xiaomi','redmi':'Xiaomi',
    'poco':'Xiaomi','oppo':'OPPO','vivo':'Vivo','realme':'Realme','huawei':'Huawei','nokia':'Nokia',
    'vsmart':'VSmart','asus':'Asus','sony':'Sony','oneplus':'OnePlus',
}

def detect_brand(text):
    low = str(text).lower()
    for kw, b in BRAND_GAZETTEER.items():
        if kw in low: return b
    return 'Unknown'

train_df['brand'] = train_df['comment'].apply(detect_brand)
print('Brand distribution:', dict(Counter(train_df['brand']).most_common()))
for q in ['camera chụp đêm có tốt không','pin dùng được bao lâu','giá có đắt quá không','nhân viên tư vấn nhiệt tình']:
    print(f'  {q!r:45s} -> {aspect_from_query(q)}')

## 4. NER + Gazetteer (trung thực với domain)

`underthesea.ner` là NER tiếng Việt tổng quát (PER/LOC/ORG) → bắt tốt **thương hiệu/cửa hàng**.
Thực thể e-commerce (dòng máy, sản phẩm) không nằm trong nhãn NER tổng quát → bổ sung **gazetteer**.
`PhoNER_COVID19` là NER domain COVID, **không phù hợp** review điện thoại nên không dùng để gán nhãn sản phẩm.

In [ ]:
from underthesea import ner as uts_ner
from tqdm.auto import tqdm

def extract_entities(text):
    text = str(text); ents = []
    try:
        for tok in uts_ner(text[:400]):
            word, pos, chunk_tag, ner_tag = tok
            if ner_tag != 'O':
                ents.append({'text': word, 'type': ner_tag.split('-')[-1], 'source': 'ner'})
    except Exception:
        pass
    low = text.lower()
    for kw, brand in BRAND_GAZETTEER.items():
        if kw in low:
            ents.append({'text': brand, 'type': 'BRAND', 'source': 'gazetteer'})
    return ents

demo = 'iPhone 15 Pro Max của Apple chụp ảnh đẹp, mua tại Thế Giới Di Động giá 30 triệu'
print('Demo:', demo)
for e in extract_entities(demo): print('  ', e)

sample = train_df.head(800)
ents_all = [extract_entities(c) for c in tqdm(sample['comment'], desc='NER(sample)')]
ent_types = Counter(e['type'] for es in ents_all for e in es)
print('\nEntity types (800 mau):', dict(ent_types))

## 5. Embedding — xây corpus căn chỉnh với nhãn vàng

In [ ]:
mask = train_df['clean_text'].str.len() > 5
corpus_df = train_df[mask].reset_index(drop=True)
corpus         = corpus_df['clean_text'].tolist()
corpus_raw     = [seg(c) for c in tqdm(corpus_df['comment'], desc='seg(PhoBERT)')]
corpus_aspects = [set(a) for a in corpus_df['aspects']]   # ground-truth / document
print(f'Corpus: {len(corpus)} documents (can chinh voi gold aspects)')

### 5.1 TF-IDF — *Lec02 Word Representation* (baseline)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tfidf = TfidfVectorizer(max_features=10000)
tfidf_matrix = tfidf.fit_transform(corpus)
print('TF-IDF:', tfidf_matrix.shape)
def rank_tfidf(query, k):
    qv = tfidf.transform([preprocess_vietnamese(query)])
    s = cosine_similarity(qv, tfidf_matrix).flatten()
    return s.argsort()[::-1][:k]

### 5.2 Word2Vec — *Lec02 Word Representation*

In [ ]:
from gensim.models import Word2Vec
tok_corpus = [d.split() for d in corpus]
w2v = Word2Vec(tok_corpus, vector_size=100, window=5, min_count=2, workers=4, sg=1, epochs=20)
print('W2V vocab:', len(w2v.wv))
def doc2vec_w2v(doc):
    ws = [w for w in doc.split() if w in w2v.wv]
    return np.mean([w2v.wv[w] for w in ws], axis=0) if ws else np.zeros(w2v.vector_size)
w2v_vecs = np.vstack([doc2vec_w2v(d) for d in corpus]).astype('float32')
def rank_w2v(query, k):
    qv = doc2vec_w2v(preprocess_vietnamese(query)).reshape(1, -1)
    s = cosine_similarity(qv, w2v_vecs).flatten()
    return s.argsort()[::-1][:k]
for w in ['camera','pin','màn_hình','giá']:
    if w in w2v.wv: print(f'  {w} -> {[x for x,_ in w2v.wv.most_similar(w, topn=5)]}')

### 5.3 GloVe-SVD — *Lec02 Word Representation* (count-based)

> ⚠️ Trung thực: đây là **xấp xỉ** kiểu GloVe — SVD trên ma trận đồng xuất hiện ở **mức tài liệu**,
> không phải GloVe cửa sổ ngữ cảnh gốc. Vẫn minh hoạ embedding count-based/global.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000)
cm = cv.fit_transform(corpus)
cooc = (cm.T @ cm).astype(float); cooc.setdiag(0); cooc.eliminate_zeros()
cooc.data = np.log1p(cooc.data)
svd = TruncatedSVD(n_components=100, random_state=42)
glove_emb = svd.fit_transform(cooc)
vocab = cv.get_feature_names_out(); w2i = {w: i for i, w in enumerate(vocab)}
def doc2vec_glove(doc):
    ids = [w2i[w] for w in doc.split() if w in w2i]
    return np.mean(glove_emb[ids], axis=0) if ids else np.zeros(100)
glove_vecs = np.vstack([doc2vec_glove(d) for d in corpus]).astype('float32')
def rank_glove(query, k):
    qv = doc2vec_glove(preprocess_vietnamese(query)).reshape(1, -1)
    s = cosine_similarity(qv, glove_vecs).flatten()
    return s.argsort()[::-1][:k]
print('GloVe-SVD vectors:', glove_vecs.shape)

### 5.4 Subword Models — *Lec03*

> Word/one-hot giả định từ vựng cố định → gặp **từ hiếm / OOV** là bó tay.
> Mô hình **subword (BPE)** tách từ thành mảnh nhỏ hơn nên biểu diễn được cả từ chưa thấy. PhoBERT dùng BPE.

In [ ]:
from transformers import AutoTokenizer as _AutoTok
sub_tok = _AutoTok.from_pretrained('vinai/phobert-base-v2')
print('Tu -> cac subword token (BPE):')
for w in ['điện_thoại', 'pin', 'sạc_nhanh', 'chống_nước', 'camera', 'Galaxy', 'ngotngao']:
    print(f'  {w:16} -> {sub_tok.tokenize(w)}')
sent = seg('điện thoại pin trâu sạc nhanh chống nước camera nét')
pieces = sub_tok.tokenize(sent)
print(f'\nCau: {sent}')
print(f'-> {len(pieces)} subword: {pieces}')

### 5.5 PhoBERT — *Lec06 Transformer* (SOTA)

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
tok = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
phobert = AutoModel.from_pretrained('vinai/phobert-base-v2').to(device).eval()

@torch.no_grad()
def encode_phobert(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
        h = phobert(**enc).last_hidden_state
        m = enc['attention_mask'].unsqueeze(-1)
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.vstack(out).astype('float32')

phobert_vecs = encode_phobert(corpus_raw)
print('PhoBERT vectors:', phobert_vecs.shape)
def rank_phobert(query, k):
    qv = encode_phobert([seg(query)])
    s = cosine_similarity(qv, phobert_vecs).flatten()
    return s.argsort()[::-1][:k]

### 5.6 ⭐ So sánh công bằng: Precision@k & MRR

> FIX #5: bỏ "avg cosine" (không so sánh được giữa các không gian embedding khác nhau).
> Ground-truth: review *liên quan* tới truy vấn về aspect X nếu **nhãn vàng** của review chứa X.

In [ ]:
EVAL_QUERIES = [
    ('camera chụp ảnh có đẹp không', 'CAMERA'),
    ('pin trâu dùng được bao lâu', 'BATTERY'),
    ('màn hình hiển thị có sắc nét không', 'SCREEN'),
    ('máy chạy có mượt hiệu năng mạnh không', 'PERFORMANCE'),
    ('giá cả có hợp lý hay đắt không', 'PRICE'),
    ('thiết kế đẹp cầm có thoải mái không', 'DESIGN'),
    ('bộ nhớ lưu trữ dung lượng bao nhiêu', 'STORAGE'),
    ('loa âm thanh và các tính năng', 'FEATURES'),
    ('nhân viên tư vấn dịch vụ bảo hành', 'SER&ACC'),
]
RANKERS = {'TF-IDF': rank_tfidf, 'Word2Vec': rank_w2v, 'GloVe-SVD': rank_glove, 'PhoBERT': rank_phobert}

def precision_at_k(ranked, asp, k): return sum(1 for i in ranked[:k] if asp in corpus_aspects[i]) / k
def mrr(ranked, asp):
    for r, i in enumerate(ranked, 1):
        if asp in corpus_aspects[i]: return 1.0 / r
    return 0.0

rows = []
for name, fn in RANKERS.items():
    p5 = p10 = mr = 0.0
    for q, asp in EVAL_QUERIES:
        ranked = list(fn(q, 50))
        p5 += precision_at_k(ranked, asp, 5); p10 += precision_at_k(ranked, asp, 10); mr += mrr(ranked, asp)
    n = len(EVAL_QUERIES)
    rows.append({'Method': name, 'P@5': p5/n, 'P@10': p10/n, 'MRR': mr/n})
metrics_df = pd.DataFrame(rows).set_index('Method').round(4)
print(metrics_df)

In [ ]:
ax = metrics_df.plot(kind='bar', figsize=(10, 5))
ax.set_title('So sanh Embedding — Precision@k & MRR (nhan aspect la ground-truth)')
ax.set_ylabel('Score'); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig('fig_embedding_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

print('\n% ===== LaTeX (dan vao bao cao) =====')
print(r'\begin{tabular}{lrrr}\toprule')
print(r'Phuong phap & P@5 & P@10 & MRR \\ \midrule')
for m, r in metrics_df.iterrows():
    print(f"{m} & {r['P@5']:.3f} & {r['P@10']:.3f} & {r['MRR']:.3f} " + r'\\')
print(r'\bottomrule\end{tabular}')

## 6. RNN — *Lec05* — BiLSTM phân loại đa nhãn aspect

> Lec05 (Recurrent Neural Networks): mô hình hoá **chuỗi từ** bằng BiLSTM. Ở đây huấn luyện phân loại
> đa nhãn 10 aspect từ review (đánh giá micro-F1 trên tập dev). Mô hình này có thể gán aspect cho
> review Shopee (vốn không có nhãn) để **làm giàu Knowledge Graph** — nối Lec05 với pipeline.

In [ ]:
import torch
import torch.nn as nn
from collections import Counter as _Counter
from sklearn.metrics import f1_score

# vocab từ corpus đã tách từ
_tok_docs = [str(d).split() for d in train_df['clean_text']]
_freq = _Counter(w for doc in _tok_docs for w in doc)
itos = ['<pad>', '<unk>'] + [w for w, _c in _freq.most_common(8000)]
stoi = {w: i for i, w in enumerate(itos)}
MAXLEN = 40

def _encode(text):
    ids = [stoi.get(w, 1) for w in str(text).split()[:MAXLEN]]
    return ids + [0] * (MAXLEN - len(ids))

def _multihot(aspects):
    v = torch.zeros(len(ASPECTS))
    for a in aspects:
        if a in ASPECTS:
            v[ASPECTS.index(a)] = 1.0
    return v

dev_t = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Xtr = torch.tensor([_encode(t) for t in train_df['clean_text']]).to(dev_t)
Ytr = torch.stack([_multihot(a) for a in train_df['aspects']]).to(dev_t)
Xdv = torch.tensor([_encode(t) for t in dev_df['clean_text']]).to(dev_t)
Ydv = torch.stack([_multihot(a) for a in dev_df['aspects']]).to(dev_t)

class BiLSTMAspect(nn.Module):
    def __init__(self, vocab, emb=100, hid=128, n_aspect=len(ASPECTS)):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hid * 2, n_aspect)
    def forward(self, x):
        o, _ = self.lstm(self.emb(x))
        return self.fc(o.mean(1))

rnn = BiLSTMAspect(len(itos)).to(dev_t)
opt = torch.optim.Adam(rnn.parameters(), lr=1e-3)
lossf = nn.BCEWithLogitsLoss()
bs = 128
for ep in range(5):
    rnn.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), bs):
        idx = perm[i:i+bs]; opt.zero_grad()
        loss = lossf(rnn(Xtr[idx]), Ytr[idx]); loss.backward(); opt.step()
    rnn.eval()
    with torch.no_grad():
        pred = (torch.sigmoid(rnn(Xdv)) > 0.5).cpu().numpy()
    f1 = f1_score(Ydv.cpu().numpy(), pred, average='micro', zero_division=0)
    print(f'epoch {ep+1}/5 — dev micro-F1 = {f1:.4f}')
rnn_f1 = float(f1)
print(f'BiLSTM (Lec05) xong — micro-F1 = {rnn_f1:.4f}')

# === DEPLOY: luu model da train thanh artifact cho repo src/vngraphrag ===
os.makedirs('artifacts', exist_ok=True)
torch.save({'state_dict': {k: v.cpu() for k, v in rnn.state_dict().items()},
            'itos': itos, 'aspects': ASPECTS, 'maxlen': MAXLEN, 'emb': 100, 'hid': 128},
           'artifacts/aspect_clf.pt')
print('Saved trained model -> artifacts/aspect_clf.pt  (deploy: API POST /classify)')

## 7. Lưu kết quả cho Part 2 & báo cáo

In [ ]:
keep = [c for c in ['index','comment','n_star','label','parsed_labels','aspects','sentiments','clean_text','brand'] if c in train_df.columns]
train_df[keep].to_csv('data/train_processed.csv', index=False)
results = {
    'n_train': int(len(train_df)),
    'aspect_counts': dict(aspect_counts),
    'embedding_metrics': metrics_df.reset_index().to_dict(orient='records'),
    'ner_entity_types': dict(ent_types),
    'rnn_aspect_f1': rnn_f1,
}
with open('results_part1.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('Saved: data/train_processed.csv + results_part1.json')
print(json.dumps(results['embedding_metrics'], ensure_ascii=False, indent=2))